<a href="https://colab.research.google.com/github/Iberasa3/Hollow-Hornet/blob/main/Avispas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd

# El separador '\t' le dice a pandas que use el tabulador
df = pd.read_csv('avispas_avistamientos.csv', sep='\t') #Duh el separador era esta shit

# Ahora sí, vamos a comprobar que se han separado
print(f"Número de columnas: {df.shape[1]}")
print(f"Nombres de las columnas: {df.columns.tolist()[:5]}...")

/tmp/ipykernel_372/551559319.py:4: DtypeWarning: Columns (14,16,17,36,37,38,39,40,41,43,44,45,46,48) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('avispas_avistamientos.csv', sep='\t') #Duh el separador era esta shit


Número de columnas: 50
Nombres de las columnas: ['gbifID', 'datasetKey', 'occurrenceID', 'kingdom', 'phylum']...


Efectivamente este es mi proyecto de avispas, de momento no tengo mucho más que añadir. Cuando termine de limpiar este dataset proyectaré las coordenadas en GEE

Un par de apuntes para mí mismo, probablemente acabemos considerando a todas las subespecies de avispa asiatica como una sola para no tener que liarnos entre las distintas subespecies.

Varios factores que tengo que tener en cuenta para poder filtrar bien los datos que se me están presentando:

1- Los datos anteriores a 2005, si los hay, no hay que tenerlos en cuenta porque Vespa velutina entró a europa en 2004

2- Cuidado con null island (hecho)

3- cuidado con los rangos en los que el avistamiento tiene gran índice de error (hecho)

4- cuidado con los avistamientos no-salvajes (hecho, preservados eliminados)

5- hay que chekiar las diferentes subespecies por si acaso

6- hay que chekiar solo los que sean de europa (hecho)


In [3]:
df['infraspecificEpithet'].unique() #Si la celda está vacía entonces asumimos que no pertenece a ninguna subespecie o que es 'común'

array([nan, 'nigrithorax', 'variana', 'auraria', 'flavitarsus',
       'velutina', 'karnyi', 'celebensis', 'ardens', 'sumbana',
       'divergens', 'floresiana', 'timorensis'], dtype=object)

In [4]:
paises_europa = ['ES', 'FR', 'PT', 'BE', 'IT', 'DE', 'CH', 'LU', 'GB', 'IE', 'NL'] #El análisis se basa en europa así que nos quedamos solo con los países europeos. La mayoría de registros de todas formas son europeos.
df_europa_0 = df[df['countryCode'].isin(paises_europa)]
print(f"Pasamos de {len(df)} a {len(df_europa_0)} registros.")

Pasamos de 377899 a 372895 registros.


In [5]:
#Quitamos todas las coordenadas basura y los nulls
df_coords0 = df[(df['decimalLatitude'] != 0.0) & (df['decimalLongitude'] != 0.0)]
df_coords0 = df_coords0.dropna(subset=['decimalLatitude', 'decimalLongitude'])

print(f"Registros originales: {len(df)}")
print(f"Registros después de haber quitado los nulls y los null-islands: {len(df_coords0)}")

Registros originales: 377899
Registros después de haber quitado los nulls y los null-islands: 376580


In [6]:
#Quitamos las instancias en las que la incertidumbre es superior a 1km. Habría que ver por qué coño existen incertidumbres tan grandes
umbral_maximo = 1000
df_fin = df_coords0[(df_coords0['coordinateUncertaintyInMeters'] <= 1000) & (df['year'] >= 2005)]

/tmp/ipykernel_372/340826367.py:3: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_fin = df_coords0[(df_coords0['coordinateUncertaintyInMeters'] <= 1000) & (df['year'] >= 2005)]


Hay que tener cuidado con las instancias que se hayan tomado en el mismo instante, puede ser que un pibe haya apuntado varias veces a una misma colmena o incluso a una misma avispa, y que por ello haya zonas sobrerepresentadas.
Ahora habría que ver cómo chota hacemos esto del spatial thinning.
Como hemos descargado la versión simple del Darwin Core igual tenemos datos que nos ayudan con esto. De hecho igual en un futuro lo hago directamente con DWC

In [9]:
df_fin = df_fin[df_fin['basisOfRecord'] != 'PRESERVED_SPECIMEN']
print(df_fin['basisOfRecord'].unique())

array(['HUMAN_OBSERVATION', 'PRESERVED_SPECIMEN'], dtype=object)

Vamos con el spatial thinning.

Podemos aplicar una agrupación por celda, útil para evitar sobremuestreos pero falla para detectar zonas más idóneas en las que las avispas hayan podido construir sus colmenas. Luego podemos sumarlo a un check de correlación para ver si nos ha salido bien.

La cosa es que si vamos a aplicar un random forest se asume que cada dato va a ser independiente, el modelo no va a saber que los distintos puntos que están cerca entre sí pueden reflejar una colmena próxima. Además al reflejar los puntos en el propio mapa la resolución no va a dar para más, así que creo que hacer el spatial thinning sin tener en cuenta la posibilidad de colmena es, de momento, suficiente. Esto es un modelo de distribución, no uno de abundancia.

¿CUÁNTO SE ALEJAN LAS AVISPAS DE SUS NIDOS?

